# 보 내부력 시뮬레이터: 시뮬레이션 데이터 분석 (Google Colab)

왼쪽 파일 패널에 `beam.py`와 `sweep.py`를 업로드한 뒤 위에서부터 차례로 실행하세요.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from beam import solve_beam
from sweep import make_sweep, analyze

## 1. 기준 예제 검산
L = 6 m, 강재(7,850 kg/m³), 단면 0.2×0.4 m, a = 2 m, P = 10,000 N

In [ ]:
r = solve_beam(6, 7850, 0.2, 0.4, 2, 10000)
print(f"R_A = {r['R_A_N']:,.2f} N, R_B = {r['R_B_N']:,.2f} N")
print(f"Mmax = {r['M_max_Nm']:,.2f} N·m at x = {r['x_Mmax_m']:.3f} m")

fig, (axV, axM) = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
axV.plot(r['x'], r['V']); axV.axhline(0, color='k', lw=0.8); axV.set_ylabel('V [N]'); axV.set_title('SFD')
axM.plot(r['x'], r['M'], color='tab:red'); axM.axhline(0, color='k', lw=0.8); axM.set_ylabel('M [N·m]'); axM.set_title('BMD')
axM.plot(r['x_Mmax_m'], r['M_max_Nm'], 'ko'); axM.set_xlabel('x [m]')
plt.tight_layout(); plt.show()

## 2. 데이터셋 생성 (1,512행)과 분석

In [ ]:
df = make_sweep()
df.to_csv('sweep.csv', index=False, encoding='utf-8-sig')
df.head()

In [ ]:
analyze(df)

## 3. 시각화

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# a/L - Mmax (재료별)
df.pivot_table(index='a_ratio', columns='material', values='M_max_Nm', aggfunc='mean').plot(ax=axes[0], marker='o')
axes[0].set_title('Mean Mmax by load position'); axes[0].set_xlabel('a/L'); axes[0].set_ylabel('Mmax [N·m]')

# 히트맵: a/L x L (강재, P = 10 kN)
h = df[(df.material == 'steel') & (df.P_N == 10000)].pivot_table(index='L_m', columns='a_ratio', values='M_max_Nm')
im = axes[1].imshow(h.values, origin='lower', aspect='auto', cmap='Reds',
                    extent=[h.columns.min(), h.columns.max(), h.index.min(), h.index.max()])
fig.colorbar(im, ax=axes[1], label='Mmax [N·m]')
axes[1].set_title('Mmax heatmap (steel, P = 10 kN)'); axes[1].set_xlabel('a/L'); axes[1].set_ylabel('L [m]')

# 재료별 자중 비중
df.groupby('material')['selfweight_share'].mean().sort_values().plot.bar(ax=axes[2])
axes[2].set_title('Self-weight share of Mmax'); axes[2].set_ylabel('wL²/8 ÷ Mmax')
plt.tight_layout(); plt.show()